# 02 — Regression Model

Predict the **actual crowd count** (continuous value) using LinearRegression and compare with other regressors.

In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')

from apps.predict.ml.features import engineer_features
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

df = pd.read_csv('../data/raw/ahmedabad_metro_bookings.csv')
X = engineer_features(df)
y = df['actual_crowd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Compare regression models
models = {
    'LinearRegression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1),
    'RandomForest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    results.append({'Model': name, 'R2': r2, 'RMSE': rmse, 'MAE': mae})
    print(f'{name:25s} | R2={r2:.4f} | RMSE={rmse:.2f} | MAE={mae:.2f}')

results_df = pd.DataFrame(results)
results_df

In [ ]:
# Visualize predictions vs actuals for best model
best_model = GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42)
best_model.fit(X_train_s, y_train)
y_pred = best_model.predict(X_test_s)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(y_test, y_pred, alpha=0.5, color='#6366f1')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
axes[0].set_xlabel('Actual Crowd')
axes[0].set_ylabel('Predicted Crowd')
axes[0].set_title('Actual vs Predicted')

# Residuals
residuals = y_test - y_pred
axes[1].hist(residuals, bins=25, color='#8b5cf6', edgecolor='white')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validation for LinearRegression
lr = LinearRegression()
cv_scores = cross_val_score(lr, scaler.fit_transform(X), y, cv=5, scoring='r2')
print(f'LinearRegression 5-fold CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Per-fold: {cv_scores}')